# Import Library

In [1]:
import pandas as pd
import os
import opendatasets as od
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
import random

In [2]:
# set random seed
seed=3
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Load Data

In [ ]:
# Prepare dataset
def list_all_dataset(dir: str, labels: list):

    genre_name=[]
    audio_name=[]
    audio_length=[]
    amplitude=[]
    amplitude_avg=[]
    sampling_rate=[]
    file_path=[]

    for _, label in enumerate(labels):
        class_dir = os.path.join(dir, label)
        for file in os.listdir(class_dir):
            if file.endswith(".wav"):
                audio_path = os.path.join(class_dir, file)

                genre_name.append(class_dir.split("/")[-1])
                audio_name.append(audio_path.split("/")[-1])
                length = librosa.get_duration(filename=audio_path)
                y, sr = librosa.load(audio_path)
                amplitude.append(y)
                amplitude_avg.append(y.mean())
                sampling_rate.append(sr)
                audio_length.append(length)
                file_path.append(audio_path)

    df = pd.DataFrame({
        "Label":genre_name,
        "Audio":audio_name,
        "Length":audio_length,
        "Amplitude":amplitude,
        "Amplitude avg":amplitude_avg,
        "Sampling rate":sampling_rate,
        "File path":audio_path
    }).sort_values(['Label','Audio']).reset_index(drop=True)

    return df

In [3]:
dataset = 'https://www.kaggle.com/datasets/andrewmvd/cat-meow-classification'
od.download(dataset)

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/cat-meow-classification


100%|██████████| 12.4M/12.4M [00:06<00:00, 2.09MB/s]


In [ ]:
dir = '/Users/fadilahnurimani/Documents/Projects/cat-meow-audio-classifier/cat-meow-classification/dataset'
dataset_path = sorted(os.listdir(dir))
df = list_all_dataset(dir, genre_labels)

# Preprocessing Functions

In [ ]:
# Extract MFCC and Chroma features
def extract_features(audio_path: str, approach: str = 'stacked', method: str = 'rms', sr: int = 22050, n_mfcc: int = 13, n_chroma: int = 12):
    y, _ = librosa.load(audio_path, sr=sr)

    # Initialize feature variables
    mfccs = None
    chroma = None

    # Extract features based on method
    if method=='rms'
        chroma = librosa.feature.rms(y=y)
    if method=='mfcss'
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    if method=='chroma_stft'
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)

    # Handle feature stacking or returning individual features
    if method == 'stacked':
        features = np.vstack([mfccs, chroma])
    elif method == 'mfccs':
        features = mfccs
    elif method == 'chroma':
        features = chroma
    else:
        raise ValueError("Invalid method. Choose from 'stacked', 'mfccs', or 'chroma'.")

    return features


# EDA